## Compare several "Composite" portfolio
### "Composite" portfolio : `[CorrClusterSelector, DualMomentumSelector, "Optimizer"]`
It is a `ModelPipeline` of 2 Selctors and 1 optimizer
where "Optimizer" are instances of `CVaRAnalyzer`, `KellyEngine`, "EWP", `SDAnalyzer`, `MADAnalyzer`, and `LSDAnalyzer`

In [1]:
import numpy as np
import pandas as pd

import azapy as az
print(f"azapy version {az.version()} >= 1.2.0", flush=True)

azapy version 1.2.6 >= 1.2.0


### Collect market data

In [2]:
mktdir = '../MkTdata'
sdate = '2012-01-01'
edate = 'today'

symb = ['GLD', 'TLT', 'IHI', 'SPY', 'OIH',
        'XAR', 'XBI', 'XHE', 'XHS', 'XLB',
        'XLE', 'XLF', 'XLI', 'XLK', 'XLU', 
        'XLV', 'XLY', 'XRT', 'SPY', 'ONEQ', 
        'QQQ', 'DIA', 'ILF', 'XSW', 'PGF', 
        'IDV', 'JNK', 'HYG', 'SDIV', 'VIG', 
        'SLV', 'AAPL', 'MSFT', 'AMZN', 'GOOG', 
        'IYT', 'IWM', 'BRK-B', 'ITA', 'VUG']

mktdata = az.readMkT(symb, sdate=sdate, edate=edate, file_dir=mktdir, 
                     verbose=False)

### Set general parameters for Selectors and Port_Generator

In [3]:
corr_threshold = 0.98
freq = 'Q'

nw = 5
ths = np.floor(len(symb) * 0.3)

hlength = 1.25
rtype = 'Sharpe'
mu0 = 0.01

histoffset = hlength
fixoffset = -1

### "Optimizer" = CVaRAnalyzer

In [4]:
pname = 'CVaR'
ccs = az.CorrClusterSelector(corr_threshold=corr_threshold, freq=freq)
dms = az.DualMomentumSelector(nw=nw, threshold=ths)
alpha = [0.95, 0.9]
cvar = az.CVaRAnalyzer(alpha=alpha, freq=freq, hlength=hlength,
                       rtype=rtype, mu0=mu0)
model = az.ModelPipeline([ccs, dms, cvar])
pp_cvar = az.Port_Generator(mktdata, 
                       freq=freq, 
                       fixoffset=fixoffset, 
                       histoffset=histoffset,
                       pname=pname)
port_cvar = pp_cvar.set_model(model)
_ = pp_cvar.port_view(fancy=True)

### "Optimizer" = KellyEngine

In [5]:
pname = 'Kelly'
ccs = az.CorrClusterSelector(corr_threshold=corr_threshold, freq=freq)
dms = az.DualMomentumSelector(nw=nw, threshold=ths)
kelly = az.KellyEngine(freq=freq, hlength=hlength)
model = az.ModelPipeline([ccs, dms, kelly])
pp_kelly = az.Port_Generator(mktdata, 
                             freq=freq, 
                             fixoffset=fixoffset, 
                             histoffset=histoffset,
                             pname=pname)
port_kelly = pp_kelly.set_model(model)
_ = pp_kelly.port_view(fancy=True)

### "Optimizer" = MADAnalyzer

In [6]:
pname = 'MAD'
ccs = az.CorrClusterSelector(corr_threshold=corr_threshold, freq=freq)
dms = az.DualMomentumSelector(nw=nw, threshold=ths)
coef = [1, 1, 1, 1]
mad = az.MADAnalyzer(coef=coef, freq=freq, hlength=hlength)
model = az.ModelPipeline([ccs, dms, mad])
pp_mad = az.Port_Generator(mktdata, 
                             freq=freq, 
                             fixoffset=fixoffset, 
                             histoffset=histoffset,
                             pname=pname)
port_mad = pp_mad.set_model(model)
_ = pp_mad.port_view(fancy=True)

### "Optimizer" = "EWP"

In [7]:
pname = 'EWP'
ccs = az.CorrClusterSelector(corr_threshold=corr_threshold, freq=freq)
nw_ewp = 3
dms = az.DualMomentumSelector(nw=nw_ewp, threshold=ths)
model = az.ModelPipeline([ccs, dms, "EWP"])
pp_ewp = az.Port_Generator(mktdata, 
                             freq=freq, 
                             fixoffset=fixoffset, 
                             histoffset=histoffset,
                             pname=pname)
port_ewp = pp_ewp.set_model(model)
_ = pp_ewp.port_view(fancy=True)

### "Optimizer" = SDAnalyzer

In [8]:
pname = 'SD'
ccs = az.CorrClusterSelector(corr_threshold=corr_threshold, freq=freq)
dms = az.DualMomentumSelector(nw=nw, threshold=ths)
sd = az.SDAnalyzer(freq=freq, hlength=hlength)
model = az.ModelPipeline([ccs, dms, sd])
pp_sd = az.Port_Generator(mktdata, 
                             freq=freq, 
                             fixoffset=fixoffset, 
                             histoffset=histoffset,
                             pname=pname)
port_sd = pp_sd.set_model(model)
_ = pp_sd.port_view(fancy=True)

### "Optimizer" = LSDAnalyzer

In [9]:
pname = 'LSD'
ccs = az.CorrClusterSelector(corr_threshold=corr_threshold, freq=freq)
dms = az.DualMomentumSelector(nw=nw, threshold=ths)
coef = [1, 1, 1, 1]
lsd = az.LSDAnalyzer(coef=coef, freq=freq, hlength=hlength)
model = az.ModelPipeline([ccs, dms, lsd])
pp_lsd = az.Port_Generator(mktdata, 
                             freq=freq, 
                             fixoffset=fixoffset, 
                             histoffset=histoffset,
                             pname=pname)
port_lsd = pp_lsd.set_model(model)
_ = pp_lsd.port_view(fancy=True)

### Compare the portfolio strategies

In [10]:
pq = az.Port_Simple([port_cvar, port_kelly, port_mad, port_ewp, port_sd, port_lsd])
_ = pq.set_model()
_ = pq.port_view_all(componly=True, fancy=True)

### Compare the overall performance of the strategies

In [11]:
pq.port_perf(componly=True, fancy=True)

,RR,DD,RoMaD,DD_date,DD_start,DD_end,DD_days
symbol,,,,,,,
SD,16.05,-27.39,0.586037,2022-05-24,2022-03-29,2022-11-11,227
MAD,16.09,-28.16,0.571518,2022-05-24,2022-01-03,2023-01-23,385
CVaR,15.23,-31.72,0.480157,2025-04-08,2025-01-21,2025-10-08,260
LSD,15.08,-31.52,0.478362,2025-04-08,2025-01-21,2025-09-25,247
EWP,16.29,-36.51,0.446326,2020-10-28,2020-06-08,2023-01-26,962
Kelly,12.94,-35.71,0.362304,2022-05-24,2020-12-28,2023-03-20,812


### Compare annual returns 

In [12]:
an_cvar = pp_cvar.port_annual_returns()
an_kelly = pp_kelly.port_annual_returns()
an_ewp = pp_ewp.port_annual_returns()
an_sd = pp_sd.port_annual_returns()
an_mad = pp_mad.port_annual_returns()
an_lsd = pp_lsd.port_annual_returns()
(an_cvar.join(an_kelly).join(an_ewp).join(an_sd).join(an_mad).join(an_lsd) * 100).round(2)

,CVaR,Kelly,EWP,SD,MAD,LSD
year,,,,,,
2013,16.93,16.93,15.82,12.60,14.65,14.51
2014,-8.46,-21.02,-9.73,-6.78,-8.42,-8.51
2015,10.51,27.66,6.85,11.98,12.63,9.87
2016,26.59,24.53,45.81,24.89,25.02,26.25
2017,16.22,-1.08,6.75,16.30,15.72,16.52
2018,17.99,23.41,21.62,15.17,12.10,18.29
2019,21.74,21.74,19.12,21.76,21.74,21.74
2020,46.11,45.98,49.67,46.13,46.08,46.11
2021,5.67,-5.29,-2.30,6.00,6.07,6.06


### Compare the rebalancing period returns

In [13]:
zz_cvar = pp_cvar.port_period_returns().drop('Dfix', axis=1).set_index('Droll')[['RR']].rename({'RR': pp_cvar.pname}, axis=1)
zz_kelly = pp_kelly.port_period_returns().drop('Dfix', axis=1).set_index('Droll')[['RR']].rename({'RR': pp_kelly.pname}, axis=1)
zz_ewp = pp_ewp.port_period_returns().drop('Dfix', axis=1).set_index('Droll')[['RR']].rename({'RR': pp_ewp.pname}, axis=1)
zz_sd = pp_sd.port_period_returns().drop('Dfix', axis=1).set_index('Droll')[['RR']].rename({'RR': pp_sd.pname}, axis=1)
zz_mad = pp_mad.port_period_returns().drop('Dfix', axis=1).set_index('Droll')[['RR']].rename({'RR': pp_mad.pname}, axis=1)
zz_lsd = pp_lsd.port_period_returns().drop('Dfix', axis=1).set_index('Droll')[['RR']].rename({'RR': pp_lsd.pname}, axis=1)
(zz_cvar.join(zz_kelly).join(zz_ewp).join(zz_sd).join(zz_mad).join(zz_lsd) * 100).round(2)

,CVaR,Kelly,EWP,SD,MAD,LSD
Droll,,,,,,
2013-06-25,9.42,9.42,13.74,9.42,9.42,9.42
2013-09-25,6.42,6.42,1.48,2.29,4.24,4.09
2013-12-26,2.37,2.37,0.90,2.56,2.47,2.48
2014-03-26,0.94,2.86,0.10,0.30,0.34,0.92
2014-06-25,1.70,1.76,1.30,1.71,1.70,1.70
2014-09-25,-13.15,-27.85,-13.13,-11.27,-12.66,-13.12
2014-12-26,2.47,4.39,3.11,2.78,2.55,2.42
2015-03-26,6.05,10.47,4.45,5.15,4.66,5.47
2015-06-25,-4.70,5.17,1.56,-3.02,-1.73,-4.82
